# Section 06: Ensuring Solution and Operations Excellence

**Companion notebook for**: `06-operations-excellence.html`

## Learning Objectives
- Configure Cloud Monitoring metrics and alerting
- Set up Cloud Logging sinks and log-based metrics
- Define SLOs and error budgets programmatically
- Design observability dashboards
- Practice incident response and chaos engineering patterns

## Prerequisites
- A Google Cloud project with billing enabled
- Python 3.10+

---
## Setup

In [ ]:
%pip install -q google-cloud-monitoring google-cloud-logging google-cloud-trace

In [ ]:
import os, json
PROJECT_ID = os.environ.get('GCP_PROJECT_ID', 'your-project-id')
REGION = 'us-central1'
print(f'Project: {PROJECT_ID}\nRegion: {REGION}')
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

---
## Section 1: Cloud Monitoring Metrics

Explore GCP metric types and build alerting policies.

In [ ]:
metric_types = [
    ('GCP System', 'compute.googleapis.com/instance/cpu/utilization', 'Automatic', '24 months'),
    ('GCP System', 'run.googleapis.com/request_count', 'Automatic', '24 months'),
    ('GCP System', 'loadbalancing.googleapis.com/https/request_count', 'Automatic', '24 months'),
    ('Custom', 'custom.googleapis.com/myapp/queue_depth', 'Application code', '24 months'),
    ('Custom', 'custom.googleapis.com/myapp/cache_hit_rate', 'Application code', '24 months'),
    ('Agent', 'agent.googleapis.com/memory/percent_used', 'Ops Agent', '24 months'),
]

print(f"{'Type':<12} {'Metric':<55} {'Source':<18} {'Retention'}")
print('-' * 100)
for t, m, s, r in metric_types:
    print(f'{t:<12} {m:<55} {s:<18} {r}')

In [ ]:
# Alerting policy examples
alert_policies = [
    {
        'name': 'High CPU',
        'metric': 'compute.googleapis.com/instance/cpu/utilization',
        'condition': '> 80% for 5 minutes',
        'severity': 'WARNING',
        'action': 'Check for traffic spike or runaway process',
    },
    {
        'name': 'High Error Rate',
        'metric': 'run.googleapis.com/request_count (5xx)',
        'condition': '> 1% of total requests for 2 minutes',
        'severity': 'CRITICAL',
        'action': 'Check recent deployment, rollback if needed',
    },
    {
        'name': 'High Latency',
        'metric': 'run.googleapis.com/request_latencies (p99)',
        'condition': '> 2000ms for 5 minutes',
        'severity': 'WARNING',
        'action': 'Check DB queries, cache hit rate, dependencies',
    },
    {
        'name': 'Disk Full',
        'metric': 'compute.googleapis.com/instance/disk/percent_used',
        'condition': '> 90% for 10 minutes',
        'severity': 'CRITICAL',
        'action': 'Expand disk, clean up old files, check log rotation',
    },
    {
        'name': 'Uptime Check Failed',
        'metric': 'monitoring.googleapis.com/uptime_check/check_passed',
        'condition': 'Failed from 2+ locations for 1 minute',
        'severity': 'CRITICAL',
        'action': 'Page on-call, check LB health, regional failover',
    },
]

print('Essential Alerting Policies')
print('=' * 80)
for p in alert_policies:
    print(f"\n  {p['name']} [{p['severity']}]")
    print(f"  Metric: {p['metric']}")
    print(f"  Condition: {p['condition']}")
    print(f"  Action: {p['action']}")

---
## Section 2: Cloud Logging Configuration

In [ ]:
log_types = [
    ('Admin Activity', 'Resource modifications', '400 days', 'No (always on)', 'Included'),
    ('Data Access', 'Data reads/metadata', '30 days', 'Yes (off by default)', 'Must enable'),
    ('System Event', 'Google-initiated actions', '400 days', 'No', 'Included'),
    ('Policy Denied', 'Security policy violations', '30 days', 'No', 'Included'),
    ('Platform Logs', 'App logs (GKE, Run)', '30 days', 'Yes', 'Included'),
]

print(f"{'Log Type':<18} {'Captures':<28} {'Retention':<12} {'Disable?':<20} {'Cost'}")
print('-' * 95)
for lt in log_types:
    print(f'{lt[0]:<18} {lt[1]:<28} {lt[2]:<12} {lt[3]:<20} {lt[4]}')

print('\nLog Sink Commands:')
sinks = [
    ('Audit to BigQuery', 'gcloud logging sinks create audit-to-bq bigquery.googleapis.com/projects/P/datasets/audit --log-filter=\'logName:"cloudaudit.googleapis.com"\' --use-partitioned-tables'),
    ('Warnings to GCS', 'gcloud logging sinks create warn-to-gcs storage.googleapis.com/log-archive --log-filter=\'severity >= WARNING\''),
    ('All to Pub/Sub', 'gcloud logging sinks create all-to-pubsub pubsub.googleapis.com/projects/P/topics/logs'),
]
for desc, cmd in sinks:
    print(f'\n## {desc}')
    print(f'   {cmd}')

---
## Section 3: SLO and Error Budget Calculator

In [ ]:
def slo_report(slo_target, period_days, good_requests, total_requests):
    """Calculate SLO compliance and error budget."""
    total_minutes = period_days * 24 * 60
    budget_minutes = total_minutes * (1 - slo_target)
    
    actual_availability = good_requests / total_requests if total_requests > 0 else 0
    budget_consumed = (1 - actual_availability) / (1 - slo_target) * 100 if slo_target < 1 else 0
    budget_remaining = max(0, 100 - budget_consumed)
    
    return {
        'slo_target': slo_target,
        'actual': actual_availability,
        'budget_total_min': budget_minutes,
        'budget_consumed_pct': min(budget_consumed, 100),
        'budget_remaining_pct': budget_remaining,
        'status': 'OK' if budget_remaining > 25 else 'WARNING' if budget_remaining > 0 else 'EXHAUSTED',
    }

# Simulate SLO reports for different services
services = [
    ('API Gateway', 0.999, 30, 9_985_000, 10_000_000),
    ('Payment Service', 0.9995, 30, 4_997_000, 5_000_000),
    ('Search Service', 0.995, 30, 1_960_000, 2_000_000),
    ('Auth Service', 0.9999, 30, 999_850, 1_000_000),
]

print('SLO Compliance Report')
print('=' * 90)
print(f"{'Service':<20} {'SLO':<8} {'Actual':<10} {'Budget Used':<14} {'Remaining':<12} {'Status'}")
print('-' * 90)
for name, target, days, good, total in services:
    r = slo_report(target, days, good, total)
    print(f"{name:<20} {r['slo_target']:.2%}  {r['actual']:.4%}  {r['budget_consumed_pct']:>8.1f}%     {r['budget_remaining_pct']:>7.1f}%     {r['status']}")

print('\nBurn Rate Alert Thresholds:')
print('  2x burn rate  --> Page on-call (budget exhausted in 15 days)')
print('  5x burn rate  --> Escalate (budget exhausted in 6 days)')
print('  10x burn rate --> Critical (budget exhausted in 3 days)')

---
## Section 4: Chaos Engineering Checklist

In [ ]:
chaos_experiments = [
    {
        'name': 'Zone Failure',
        'action': 'Drain all nodes in one zone of a regional GKE cluster',
        'hypothesis': 'Service remains available, pods reschedule to other zones',
        'rollback': 'Uncordon drained nodes',
        'tools': 'kubectl drain, GKE node pool management',
    },
    {
        'name': 'DB Failover',
        'action': 'Trigger Cloud SQL failover to standby',
        'hypothesis': 'Application reconnects within 30 seconds, no data loss',
        'rollback': 'Failover is automatic (new standby is created)',
        'tools': 'gcloud sql instances failover',
    },
    {
        'name': 'Network Partition',
        'action': 'Block traffic between two services using firewall rules',
        'hypothesis': 'Circuit breaker activates, graceful degradation, no cascade',
        'rollback': 'Delete temporary firewall rule',
        'tools': 'gcloud compute firewall-rules create/delete',
    },
    {
        'name': 'Pod Kill',
        'action': 'Randomly kill pods in a deployment',
        'hypothesis': 'K8s recreates pods, service stays within SLO',
        'rollback': 'Automatic (K8s self-healing)',
        'tools': 'kubectl delete pod, Chaos Monkey, Litmus',
    },
    {
        'name': 'Load Spike',
        'action': 'Send 10x normal traffic to the service',
        'hypothesis': 'Autoscaler provisions capacity, latency stays within SLO',
        'rollback': 'Stop load generator',
        'tools': 'Locust, k6, distributed load test on GKE',
    },
]

print('Chaos Engineering Experiments')
print('=' * 70)
for exp in chaos_experiments:
    print(f"\n  Experiment: {exp['name']}")
    print(f"  Action: {exp['action']}")
    print(f"  Hypothesis: {exp['hypothesis']}")
    print(f"  Rollback: {exp['rollback']}")
    print(f"  Tools: {exp['tools']}")

---
## Section 5: Observability Three Pillars

In [ ]:
pillars = {
    'Metrics (Cloud Monitoring)': {
        'what': 'Numerical measurements over time',
        'tells_you': 'Something IS wrong (or about to be)',
        'examples': ['CPU utilization', 'Request rate', 'Error count', 'Latency p99'],
        'gcp_tool': 'Cloud Monitoring + Cloud Monitoring dashboards',
    },
    'Logs (Cloud Logging)': {
        'what': 'Discrete events with structured data',
        'tells_you': 'WHAT happened and WHEN',
        'examples': ['Request log with status code', 'Error stack trace', 'Audit event', 'Config change'],
        'gcp_tool': 'Cloud Logging + Log Explorer + Log Analytics',
    },
    'Traces (Cloud Trace)': {
        'what': 'Request path across services with timing',
        'tells_you': 'WHERE in the request path the problem is',
        'examples': ['API -> Auth -> DB -> Cache latency breakdown', 'Slow span identification'],
        'gcp_tool': 'Cloud Trace + OpenTelemetry SDK',
    },
}

for pillar, info in pillars.items():
    print(f'\n{pillar}')
    print('=' * 50)
    print(f"  What: {info['what']}")
    print(f"  Tells you: {info['tells_you']}")
    print(f"  GCP tool: {info['gcp_tool']}")
    print(f"  Examples: {', '.join(info['examples'])}")

print('\nKey insight: Metrics detect, Logs explain, Traces locate.')

---
## Section 6: Incident Response Framework

In [ ]:
incident_phases = [
    ('1. Detect', 'Automated', [
        'Alerting policy triggers notification',
        'Uptime check fails from 2+ locations',
        'SLO burn rate exceeds threshold',
        'Customer reports issue',
    ]),
    ('2. Triage', 'Incident Commander', [
        'Determine severity (P1-P4)',
        'Assign incident commander',
        'Open communication channel (Slack/Hangouts)',
        'Notify stakeholders per severity',
    ]),
    ('3. Mitigate', 'On-call Engineer', [
        'Rollback recent deployment',
        'Scale up resources',
        'Enable failover to DR region',
        'Apply temporary fix (not root cause)',
    ]),
    ('4. Resolve', 'Engineering Team', [
        'Identify root cause',
        'Develop permanent fix',
        'Deploy through normal CI/CD',
        'Verify fix resolves the issue',
    ]),
    ('5. Post-mortem', 'Team Lead', [
        'Blameless analysis of what happened',
        'Timeline of events',
        'What went well / what to improve',
        'Action items with owners and deadlines',
    ]),
]

print('Incident Response Framework')
print('=' * 60)
for phase, owner, actions in incident_phases:
    print(f'\n  {phase} (Owner: {owner})')
    for a in actions:
        print(f'    - {a}')

---
## Section 7: Monitoring gcloud Commands

In [ ]:
monitoring_cmds = [
    ('Create uptime check', '''gcloud monitoring uptime create my-check \\
    --display-name="API Health" \\
    --resource-type=uptime-url \\
    --monitored-resource-labels=host=api.example.com \\
    --http-check-path=/health \\
    --period=60s \\
    --timeout=10s'''),
    ('Create alerting policy', '''gcloud monitoring policies create \\
    --display-name="High Error Rate" \\
    --condition-display-name="5xx > 1%" \\
    --condition-filter='metric.type="run.googleapis.com/request_count"' \\
    --condition-threshold-value=0.01 \\
    --notification-channels=projects/P/notificationChannels/123'''),
    ('Create log-based metric', '''gcloud logging metrics create api-errors \\
    --description="Count of 5xx errors" \\
    --log-filter='resource.type="cloud_run_revision" AND httpRequest.status >= 500' '''),
    ('Create SLO', '''gcloud monitoring slo create \\
    --service=my-service \\
    --display-name="Availability SLO" \\
    --goal=0.999 \\
    --rolling-period=30d'''),
]

for desc, cmd in monitoring_cmds:
    print(f'## {desc}')
    print(cmd)
    print()

---
## Summary

This notebook covered PCA Section 06:

1. **Cloud Monitoring** -- Metric types, alerting policies
2. **Cloud Logging** -- Log types, sinks, log-based metrics
3. **SLO & Error Budgets** -- Calculator, burn rate alerts
4. **Chaos Engineering** -- Experiment catalog with rollback plans
5. **Observability** -- Three pillars: metrics, logs, traces
6. **Incident Response** -- Five-phase framework
7. **Monitoring Commands** -- gcloud for uptime, alerts, SLOs

This completes the GCP Professional Cloud Architect study guide notebooks.